In [ ]:
#Setup
import pandas as pd
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">What are we going to cover?</p>

<p style="font-family: Open Sans; color: #2B6264">- How can we create a LUSID webhook notification using the Notifications API via the Python SDK?</p>

<p style="font-family: Open Sans; color: #2B6264">- What are our options for content type, authentication etc. when sending webhook notifications?</p>

<p style="font-family: Open Sans; color: #2B6264">- How can we create a 3rd party webhook notification using the Notifications API via the Python SDK?</p>

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">Creating 3rd Party Webhook Notification - Setup</p>

In [ ]:
import datetime
from typing import Optional, Any
import finbourne.sdk.services.configuration as lc

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException

# Authenticate to SDK
secrets_path = os.getenv("FBN_SECRETS_PATH")
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory_configuration = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)

configuration_sets_api = api_factory_configuration.build(lc.ConfigurationSetsApi)
display(configuration_sets_api)

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">Creating a LUSID Webhook Notification</p>

In [ ]:
import pprint
import finbourne.sdk.services.notifications as ln
import finbourne.sdk.services.notifications.models as ln_models

notifications_api = api_factory_configuration.build(ln.NotificationsApi)
subscriptions_api = api_factory_configuration.build(ln.SubscriptionsApi)
display(notifications_api)
display(subscriptions_api)

# Define subscription scope and code
subscription_scope = "FinbourneUniversity"
subscription_code = "AnyPortfolioCreatedWebhook"

In [ ]:
try:
    subscriptions_api.delete_subscription(
        scope=subscription_scope,
        code=subscription_code)
except ApiException as api_exception:
    if api_exception.status == 404:
        pass
    else:
        raise

In [ ]:
subscriptions_api.create_subscription(
    create_subscription=ln.CreateSubscription(
        id=ln.ResourceId(
            scope=subscription_scope,
            code=subscription_code,
        ),
        display_name="Any Portfolio Created",
        description="Listens to all TransactionPortfolioCreated events",
        status="Active",
        matching_pattern=ln.MatchingPattern(
            event_type="PortfolioCreated",
            filter="Body.portfolioScope eq 'FinbourneUniversity' and Body.portfolioCode startsWith 'NewTradingPortfolio'"
        )
    )
)

In [ ]:
notifications_api.create_notification(
    scope=subscription_scope,
    code=subscription_code,
    create_notification_request = ln_models.CreateNotificationRequest(
        notification_id="PortfolioCreatedWebhook",
        display_name="PortfolioCreatedWebhook",
        description="Portfolio created webhook notification",
        notification_type=ln_models.NotificationType(actual_instance=ln_models.WebhookNotificationType(
            type="Webhook",
            http_method="POST",
            url="/api/api/transactionportfolios/{{Body.portfolioScope}}/{{Body.portfolioCode}}/transactions",
            authentication_type="Lusid",
            content_type="Json",
            content=[
                {
                    "transactionId": "SeedFundsTransactionUSD",
                    "type": "FundsIn",
                    "instrumentIdentifiers": {"Instrument/default/Currency": "USD"},
                    "transactionDate": "2023-02-01T00:00:00Z",
                    "settlementDate": "2023-02-01T00:00:00Z",
                    "units": 10000000,
                    "transactionPrice": {"price": 1, "type": "Price"},
                    "totalConsideration": {"amount": 10000000, "currency": "USD"},
                    "transactionCurrency": "USD"
                },
                {
                    "transactionId": "SeedFundsTransactionGBP",
                    "type": "FundsIn",
                    "instrumentIdentifiers": {"Instrument/default/Currency": "GBP"},
                    "transactionDate": "2023-02-01T00:00:00Z",
                    "settlementDate": "2023-02-01T00:00:00Z",
                    "units": 7500000,
                    "transactionPrice": {"price": 1, "type": "Price"},
                    "totalConsideration": {"amount": 7500000, "currency": "GBP"},
                    "transactionCurrency": "GBP"
                },
                {
                    "transactionId": "SeedFundsTransactionAUD",
                    "type": "FundsIn",
                    "instrumentIdentifiers": {"Instrument/default/Currency": "AUD"},
                    "transactionDate": "2023-02-01T00:00:00Z",
                    "settlementDate": "2023-02-01T00:00:00Z",
                    "units": 25000000,
                    "transactionPrice": {"price": 1, "type": "Price"},
                    "totalConsideration": {"amount": 25000000, "currency": "AUD"},
                    "transactionCurrency": "AUD"
                }
            ]
        ))
    )
)

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">Webhook Options</p>

<p style="font-family: Open Sans; color: #2B6264">Content Types</p>
<ul>
    <li style="color: #ff5200"><p style="font-family: Open Sans; color: #2B6264">JSON (application/json)</p></li>
    <li style="color: #ff5200"><p style="font-family: Open Sans; color: #2B6264">PlainText (text/plain)</p></li>
</ul>

<p style="font-family: Open Sans; color: #2B6264">Methods</p>
<ul>
    <li style="color: #ff5200"><p style="font-family: Open Sans; color: #2B6264">POST</p></li>
    <li style="color: #ff5200"><p style="font-family: Open Sans; color: #2B6264">PUT</p></li>
    <li style="color: #ff5200"><p style="font-family: Open Sans; color: #2B6264">DELETE</p></li>
</ul>

<p style="font-family: Open Sans; color: #2B6264">Authentication Options</p>
<ul>
    <li style="color: #ff5200"><p style="font-family: Open Sans; color: #2B6264">Lusid</p></li>
    <li style="color: #ff5200"><p style="font-family: Open Sans; color: #2B6264">Bearer</p></li>
    <li style="color: #ff5200"><p style="font-family: Open Sans; color: #2B6264">Basic Auth</p></li>
</ul>

In [ ]:
try:
    configuration_sets_api.get_configuration_set(
        type="Shared",
        scope="FinbourneUniversity",
        code="jira"
    )
    
    configuration_sets_api.delete_configuration_set(
        type="Shared",
        scope="FinbourneUniversity",
        code="jira"
    )
    
except ApiException as api_exception:
    if api_exception.status == 404:
        pass
    else:
        raise

In [ ]:
config_response = configuration_sets_api.create_configuration_set(
    create_configuration_set=lc.CreateConfigurationSet(
        id=lc.ResourceId(
            scope="FinbourneUniversity",
            code="jira"
        ),
        type="Shared",
        description="Access tokens for JIRA"
    )
)

configuration_sets_api.add_configuration_to_set(
    type="Shared",
    scope="FinbourneUniversity",
    code="jira",
    create_configuration_item=lc.CreateConfigurationItem(
        key="api-token",
        value="SampleToken",
        value_type="text",
        is_secret=True,
        description="API Key for JIRA"
    )
)

configuration_sets_api.add_configuration_to_set(
    type="Shared",
    scope="FinbourneUniversity",
    code="jira",
    create_configuration_item=lc.CreateConfigurationItem(
        key="username",
        value="SampleUsername",
        value_type="text",
        is_secret=False,
        description="Username for JIRA"
    )
)

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">Creating 3rd Party Webhook Notification</p>

In [ ]:
notifications_api.create_notification(
    scope=subscription_scope,
    code=subscription_code,
    create_notification_request = ln_models.CreateNotificationRequest(
        notification_id="PortfolioCreatedJiraWebhook",
        display_name="PortfolioCreatedJiraWebhook",
        description="New Portfolio Created - Ticket",
        notification_type=ln_models.NotificationType(actual_instance=ln_models.WebhookNotificationType(
            type="Webhook",
            http_method="POST",
            url="https://acmecorp.atlassian.net/rest/api/3/issue",
            authentication_type="BasicAuth",
            authentication_configuration_item_paths={
                "username": "config://shared/FinbourneUniversity/jira/username",
                "password": "config://shared/FinbourneUniversity/jira/api-token"
            },
            content_type="Json",
            content={
                "fields": {
                    "project": {"key": "DEMO"},
                    "issuetype": {"name": "Task"},
                    "summary": "TransactionPortfolio {{body.portfolioCode}} Created!",
                    "description": {
                        "type": "doc",
                        "version": 1,
                        "content": [
                            {"type": "paragraph", "content": [{"text": "A portfolio has been created. Please make your first trades!", "type": "text"}]},
                            {"type": "paragraph", "content": [{"text": "Please close this ticket once the first trade has been placed", "type": "text"}]}
                        ]
                    },
                    "assignee": {"id": "5bfe63e1ec71bd223bbe623c"},
                    "labels": ["trades", "newPortfolio"]
                }
            }
        ))
    )
)

In [ ]:
subscriptions_api.delete_subscription(
    scope=subscription_scope,
    code=subscription_code
)

configuration_sets_api.delete_configuration_set(
    type="Shared",
    scope="FinbourneUniversity",
    code="jira"
)

<p style="font-family: Open Sans; color: #2B6264; font-weight: bold; font-size: 125%">What have we covered?</p>

<p style="font-family: Open Sans; color: #2B6264">- We created a LUSID webhook notification using the Notifications API via the Python SDK.</p>

<p style="font-family: Open Sans; color: #2B6264">- We looked at the available content types, authentication methods and request types for webhook notifications.</p>

<p style="font-family: Open Sans; color: #2B6264">- We created a 3rd party webhook notification using the Notifications API via the Python SDK.</p>